In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
df = pd.read_excel("Online Retail.xlsx")
print(df.shape)

In [ ]:
import os
print(os.getcwd())

In [ ]:
os.chdir(r"C:\Users\ram1\home-lab\projects\project5")
print(os.getcwd())

In [ ]:
df = pd.read_excel("Online Retail.xlsx")
print(df.shape)

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df["CustomerID"].nunique()

In [ ]:
print("First transaction:", df["InvoiceDate"].min())
print("Last transaction:", df["InvoiceDate"].max())

In [ ]:
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Zero or negative prices:", (df["UnitPrice"] <= 0).sum())

In [ ]:
import psycopg
print(psycopg.__version__)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS retail_transactions_raw (
            invoice_no TEXT,
            stock_code TEXT,
            description TEXT,
            quantity INTEGER,
            invoice_date TIMESTAMP,
            unit_price NUMERIC,
            customer_id NUMERIC,
            country TEXT
        );
    """)
    conn.commit()

print("Raw transactions table created successfully.")

In [ ]:
import getpass

db_password = getpass.getpass("PostgreSQL password: ")

conn = psycopg.connect(
    host="192.168.1.1",
    port=5432,
    dbname="project5_rfm",
    user="ram2",
    password=db_password
)

print("Connected to:", conn.info.dbname)

In [ ]:
import io

buffer = io.StringIO()
df.to_csv(buffer, index=False, header=False, na_rep="\\N")
buffer.seek(0)

with conn.cursor() as cur:
    with cur.copy("""
        COPY retail_transactions_raw
        FROM STDIN
        WITH (FORMAT CSV, NULL '\N')
    """) as copy:
        copy.write(buffer.getvalue())
    conn.commit()

print("Imported", len(df), "transaction rows.")

In [ ]:
import io

buffer = io.StringIO()
df.to_csv(buffer, index=False, header=False, na_rep="\\N")
buffer.seek(0)

with conn.cursor() as cur:
    with cur.copy(r"""
        COPY retail_transactions_raw
        FROM STDIN
        WITH (FORMAT CSV, NULL '\N')
    """) as copy:
        copy.write(buffer.getvalue())
    conn.commit()

print("Imported", len(df), "transaction rows.")


In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM retail_transactions_raw;")
    print("PostgreSQL rows:", cur.fetchone()[0])

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS unique_customers,
            COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customer_ids
        FROM retail_transactions_raw;
    """)
    print(cur.fetchone())

In [ ]:
import psycopg
import getpass

conn = psycopg.connect(
    host="192.168.1.1",
    port=5432,
    dbname="project5_rfm",
    user="ram2",
    password=getpass.getpass("PostgreSQL password: ")
)

print("Connected to:", conn.info.dbname)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            COUNT(*) FILTER (WHERE frequency = 1) AS one_time_customers,
            COUNT(*) FILTER (WHERE frequency > 1) AS repeat_customers,
            COUNT(*) AS total_customers
        FROM customer_rfm;
    """)
    print(cur.fetchone())

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            COUNT(*) FILTER (WHERE recency <= 30) AS active_30_days,
            COUNT(*) FILTER (WHERE recency BETWEEN 31 AND 90) AS inactive_31_90_days,
            COUNT(*) FILTER (WHERE recency > 90) AS inactive_over_90_days
        FROM customer_rfm;
    """)
    print(cur.fetchone())

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            COUNT(*) AS inactive_repeat_customers,
            ROUND(SUM(monetary), 2) AS historical_spending
        FROM customer_rfm
        WHERE recency > 90
          AND frequency > 1;
    """)
    print(cur.fetchone())

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        DROP TABLE IF EXISTS customer_segments;

        CREATE TABLE customer_segments AS
        WITH scored AS (
            SELECT
                customer_id,
                recency,
                frequency,
                monetary,
                NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
                NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
                NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
            FROM customer_rfm
        )
        SELECT *,
            CASE
                WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4
                    THEN 'VIP'
                WHEN r_score >= 3 AND f_score >= 3
                    THEN 'Loyal'
                WHEN r_score <= 2 AND f_score >= 3
                    THEN 'At Risk'
                WHEN r_score = 1 AND f_score <= 2
                    THEN 'Lost'
                ELSE 'Other'
            END AS segment
        FROM scored;
    """)
    conn.commit()

    cur.execute("""
        SELECT segment, COUNT(*)
        FROM customer_segments
        GROUP BY segment
        ORDER BY COUNT(*) DESC;
    """)

    for row in cur.fetchall():
        print(row)

In [ ]:
import psycopg
import getpass

conn = psycopg.connect(
    host="192.168.1.1",
    port=5432,
    dbname="project5_rfm",
    user="ram2",
    password=getpass.getpass("PostgreSQL password: "),
    connect_timeout=10
)

print("Connected to:", conn.info.dbname)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_name IN ('customer_rfm', 'customer_segments');
    """)
    print(cur.fetchall())

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE customer_segments AS
        WITH scored AS (
            SELECT *,
                NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
                NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
                NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
            FROM customer_rfm
        )
        SELECT *,
            CASE
                WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'VIP'
                WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal'
                WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
                WHEN r_score = 1 AND f_score <= 2 THEN 'Lost'
                ELSE 'Other'
            END AS segment
        FROM scored;
    """)
    conn.commit()

print("Customer segments created.")

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            segment,
            COUNT(*) AS customers,
            ROUND(SUM(monetary), 2) AS revenue
        FROM customer_segments
        GROUP BY segment
        ORDER BY revenue DESC;
    """)
    for row in cur.fetchall():
        print(row)

In [ ]:
import pandas as pd

segments = pd.read_sql_query(
    "SELECT * FROM customer_segments",
    conn
)

segments.to_csv(
    "customer_segments.csv",
    index=False
)

print("Exported", len(segments), "customers.")

In [ ]:
import numpy as np

expected = np.select(
    [
        (segments.r_score >= 4) & (segments.f_score >= 4) & (segments.m_score >= 4),
        (segments.r_score >= 3) & (segments.f_score >= 3),
        (segments.r_score <= 2) & (segments.f_score >= 3),
        (segments.r_score <= 1) & (segments.f_score <= 2)
    ],
    ["VIP", "Loyal", "At Risk", "Lost"],
    default="Other"
)

mismatches = segments[segments.segment != expected]

print("Customers checked:", len(segments))
print("Incorrect assignments:", len(mismatches))
print("QA result:", "PASS" if len(mismatches) == 0 else "FAIL")

In [ ]:
total_customers = len(segments)
total_revenue = segments["monetary"].sum()
average_revenue = segments["monetary"].mean()

print("Customers:", total_customers)
print("Total monetary value:", round(total_revenue, 2))
print("Average per customer:", round(average_revenue, 2))
print("Revenue by segment:")
print(segments.groupby("segment")["monetary"].sum().round(2))

In [ ]:
summary = segments.groupby("segment").agg(
    customers=("customer_id", "count"),
    revenue=("monetary", "sum")
)

summary["customer_pct"] = summary["customers"] / len(segments) * 100
summary["revenue_pct"] = summary["revenue"] / segments["monetary"].sum() * 100

print(summary.round(2).to_string())

In [ ]:
behavior = segments.groupby("segment").agg(
    customers=("customer_id", "count"),
    avg_recency_days=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_spending=("monetary", "mean")
)

print(behavior.round(2).to_string())